# Entrenar YOLO para QuibioDetation

Este notebook entrena un modelo YOLO (localización + clasificación de equipos de laboratorio en un solo paso) y lo exporta a `.tflite` listo para copiar a `app/src/main/assets/`.

**Antes de correr:** `Entorno de ejecución -> Cambiar tipo de entorno de ejecución -> GPU (T4)`.

Al final vas a descargar dos archivos: `yolo_model.tflite` y `labels.txt`. Van directo a `app/src/main/assets/` del proyecto Android (reemplazando los que puso el `DetectorProvider` como fallback).

In [ ]:
# 0) Verificar que hay GPU asignada (si dice False, repetí el paso de arriba)
import torch
print("GPU disponible:", torch.cuda.is_available(), "-", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "sin GPU")

In [ ]:
# 1) Instalar dependencias
!pip install -q ultralytics roboflow pyyaml

## 2) Descargar el dataset

Elegí **una** de las dos opciones y corré solo esa celda.

- **Opción A (recomendada):** exportaste el dataset desde Roboflow. En Roboflow, en la pestaña *Export*, elegí formato **YOLOv8**, copiá el snippet de código que te da (con tu API key, workspace, proyecto y versión) y pegalo reemplazando la celda de abajo.
- **Opción B:** subiste manualmente un `.zip` con la estructura `train/valid/test` + `data.yaml`.

In [ ]:
# 2A) Opción Roboflow (proyecto qui-bio)
# IMPORTANTE: no dejes tu API key escrita acá si vas a subir este notebook a git.
# Pegala vos mismo cada vez que corras esta celda en Colab (input() no la guarda en el notebook).
from roboflow import Roboflow

api_key = input("Pegá tu Roboflow API key: ")
rf = Roboflow(api_key=api_key)
project = rf.workspace("azambranoy-uteq-edu-ec").project("qui-bio")
version = project.version(1)
dataset = version.download("yolov8")

DATASET_DIR = dataset.location
print("Dataset descargado en:", DATASET_DIR)

In [ ]:
# 2B) Opción zip manual: subí tu dataset.zip cuando aparezca el botón, y ajustá DATASET_DIR si tu zip
# no descomprime directo en /content/dataset
# from google.colab import files
# uploaded = files.upload()  # elegí tu dataset.zip
# !unzip -q -o dataset.zip -d /content/dataset
# DATASET_DIR = "/content/dataset"
# print("Dataset en:", DATASET_DIR)

In [ ]:
# 3) Revisar las clases y el orden en que quedaron (debe coincidir con labels.txt al final)
import yaml

with open(f"{DATASET_DIR}/data.yaml") as f:
    data_yaml = yaml.safe_load(f)

print("Clases detectadas (en este orden):")
for i, name in enumerate(data_yaml["names"]):
    print(f"  {i}: {name}")

## 4) Entrenar

`yolo11n` es la versión "nano": la más liviana, pensada para exportar a celular. Con ~6-10 clases y 1000-1500 imágenes en total, 100 epochs en una T4 tarda entre 20 y 40 minutos. `patience=20` corta antes si deja de mejorar.

In [ ]:
from ultralytics import YOLO

model = YOLO("yolo11n.pt")

results = model.train(
    data=f"{DATASET_DIR}/data.yaml",
    epochs=100,
    imgsz=640,
    batch=16,
    patience=20,
    project="runs_quibio",
    name="train",
)

## 5) Validar

Mirá sobre todo `map50` (no el `map50-95`, es más exigente). Arriba de 0.7 ya sirve para la demo del curso. Si alguna clase da mal, generalmente falta variedad de fotos de esa clase puntual — revisá `runs_quibio/train/confusion_matrix.png`.

In [ ]:
metrics = model.val()
print("mAP50:", metrics.box.map50)
print("mAP50-95:", metrics.box.map)

In [ ]:
# Sanity check visual: corre el modelo sobre unas imágenes de test y muestra las cajas detectadas
import glob
from IPython.display import Image as IPImage, display

test_images = glob.glob(f"{DATASET_DIR}/test/images/*")[:5]
pred_results = model.predict(test_images, conf=0.4, save=True, project="runs_quibio", name="predict")

for path in glob.glob("runs_quibio/predict/*")[:5]:
    display(IPImage(filename=path))

## 6) Exportar a TFLite y generar `labels.txt`

Genera exactamente los dos archivos que espera `YoloDetector.kt`: `yolo_model.tflite` (float32) y `labels.txt` (una clase por línea, en el mismo orden que usó el entrenamiento).

In [ ]:
export_path = model.export(format="tflite", imgsz=640)
print("Ultralytics exportó a:", export_path)

In [ ]:
import glob, shutil

# Ultralytics guarda el .tflite float32 dentro de una carpeta *_saved_model junto a los pesos
candidates = glob.glob("runs_quibio/train/weights/*_saved_model/*float32.tflite")
src = candidates[0] if candidates else str(export_path)

shutil.copy(src, "yolo_model.tflite")

with open("labels.txt", "w") as f:
    for name in data_yaml["names"]:
        f.write(f"{name}\n")

print("Listo: yolo_model.tflite y labels.txt generados en /content")

In [ ]:
# 7) Descargar los dos archivos a tu compu
from google.colab import files

files.download("yolo_model.tflite")
files.download("labels.txt")

## 8) Último paso (en tu compu, no en Colab)

1. Copiá `yolo_model.tflite` y `labels.txt` a `app/src/main/assets/` del proyecto Android (reemplazando/agregando junto a los que ya hay).
2. Compilá y corré la app: `DetectorProvider` va a detectar `yolo_model.tflite` automáticamente y va a dejar de usar `MockEquipmentDetector`.
3. Si en el celular va lento o se traba, volvé a este notebook y probá `model.export(format="tflite", int8=True, data=f"{DATASET_DIR}/data.yaml")` para una versión cuantizada más liviana (requiere ajustar `YoloDetector` para leer entrada `uint8` en vez de `float32`).